# 歌词创作者（Day 1 小练习）

## 练习目标（理念）

用 OpenAI Chat Completions：给定**流派（genre）**和**主题摘要（summary）**，生成格式化的英文歌词。

这是 Day 1 的「提示词 + messages + 一次补全」最小闭环，适合练手。

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| `load_dotenv` / API Key | 从 `.env` 读 `OPENAI_API_KEY` 并做前缀体检 |
| system / user messages | system 定「会写歌词」，user 拼流派 + 摘要 |
| `chat.completions.create` | 单次调用 `gpt-4.1-mini` |
| 后处理输出 | 把字面量 `\n` 转成真正换行再打印 |

## 怎么跑

1. 准备 `.env`，放入以 `sk-proj-` 开头的 OpenAI API Key
2. 从上到下运行单元格；可在「创建消息」格改 `theme` / `summary`
3. 最后一格调用 `summarize(response)` 查看歌词


In [7]:
# ========== 导入：后面创建客户端、读环境变量都靠它们 ==========

# 导入标准库 os：读环境变量里的 API Key
import os
# 从 dotenv 导入 load_dotenv：把 .env 读进进程环境
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类
from openai import OpenAI


In [8]:
# ========== 环境体检 + 创建 OpenAI 客户端 ==========

# 加载 .env；override=True 用文件值覆盖已有同名环境变量
load_dotenv(override=True)
# 读取 OPENAI_API_KEY（没有则得到 None）
api_key = os.getenv('OPENAI_API_KEY')

# 钥匙体检：缺失 / 前缀不对 / 首尾空白 → 打印英文排障提示（保留原文）
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

# 用体检过的 api_key 创建官方 OpenAI 客户端（默认走 api.openai.com）
client = OpenAI(api_key=api_key)


API key found and looks good so far!


In [9]:
# ========== 提示词模板：system 定角色，user 预留「流派+摘要」前缀 ==========

# system_prompt：告诉模型要按歌曲/诗歌格式写歌词（英文指令保留，改译会改变输出风格）
system_prompt = """
    You are an assistant that can create song lyrics based off a genere and a summary for a theme of a song.
    When creating the song lyrics, format them in a typical song/poem like format.
"""
# user_prompt：用户消息的固定前缀；后面会拼接 theme 与 summary
user_prompt = """
    Here is the genere and short summary of the song to create lyrics for.
"""


In [10]:
# ========== 创建 messages：把流派与主题摘要拼进 user 内容 ==========

# 流派字符串（可改成 pop / hip-hop 等做实验）
theme = 'rock'
# 主题一句话摘要（可改成你自己的故事线）
summary = 'driving along an unknown road, thinking about the future.'

# Chat Completions 标准 messages 列表：先 system，再 user
messages = [
    {"role": "system", "content": system_prompt},
    # 注意：原逻辑是字符串直接拼接 theme + summary，中间没有分隔符，保持原样
    {"role": "user", "content": user_prompt + theme + summary}
]


In [ ]:
# ========== 格式化输出：把模型可能返回的字面量 \n 变成真换行 ==========

def summarize(response):
    # 从 Chat Completions 响应里取出助手正文；若为 None 则当成空串
    raw_msg = response.choices[0].message.content or ""

    # 有的模型会返回字面量反斜杠-n；先处理 \n\n 再处理 \n，避免双重替换搞乱段落
    formatted = raw_msg.replace("\\n\\n", "\n\n").replace("\\n", "\n")

    # 打印到笔记本 stdout，便于阅读歌词分行
    print(formatted)


In [12]:
# ========== 调用 OpenAI：单次非流式补全 ==========

# model id 与 messages 一起发给 API；返回完整 response 对象供下一格打印
response = client.chat.completions.create(model="gpt-4.1-mini", messages=messages)


In [ ]:
# ========== 打印响应：走上面的 summarize 做换行清理 ==========

summarize(response)


## 小提示

改 `theme` / `summary` 后，从「创建消息」那一格重新往下跑即可换一首歌。  
若输出仍挤在一行，检查 `summarize` 是否把字面量 `\n` 转成了真正换行。
